# PART D & E: FULL RAG PIPELINE & CRITICAL EVALUATION
**Student:** Dabone Abdoul Latif  
**Index:** 10012200015

---

## 🔵 PART D: FULL RAG PIPELINE IMPLEMENTATION
**Objective:** Build a complete pipeline (Query -> Retrieval -> Context -> Prompt -> LLM -> Response) with staged logging.

In [ ]:
import json
import faiss
import os
import numpy as np
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from groq import Groq
from dotenv import load_dotenv

load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

# Load Assets
with open('chunks/csv_chunks.json', 'r') as f: csv_chunks = json.load(f)
with open('chunks/pdf_chunks.json', 'r') as f: pdf_chunks = json.load(f)
all_chunks = csv_chunks + pdf_chunks
model = SentenceTransformer('all-MiniLM-L6-v2')
index = faiss.read_index('indexes/rag_index.faiss')
texts = [c['text'] for c in all_chunks]
bm25 = BM25Okapi([t.lower().split() for t in texts])

In [ ]:
def rag_pipeline(query, k=3, verbose=True):
    if verbose: print(f"[LOG] Starting pipeline for query: '{query}'")
    
    # Stage 1: Retrieval
    query_vec = model.encode([query]).astype('float32')
    distances, indices = index.search(query_vec, k * 2)
    bm25_scores = bm25.get_scores(query.lower().split())
    max_bm25 = max(bm25_scores) if max(bm25_scores) > 0 else 1
    
    combined = []
    for i, chunk in enumerate(all_chunks):
        v_score = 0
        for rank, idx in enumerate(indices[0]):
            if idx == i: v_score = 1 / (1 + distances[0][rank])
        score = (0.5 * v_score) + (0.5 * (bm25_scores[i] / max_bm25))
        if score > 0: combined.append({'chunk': chunk, 'score': score})
    
    results = sorted(combined, key=lambda x: x['score'], reverse=True)[:k]
    
    if verbose:
        print("[LOG] Stage 1: Retrieved Documents & Scores")
        for i, res in enumerate(results):
            print(f"      - {res['chunk']['source']} | Score: {res['score']:.4f} | Text: {res['chunk']['text'][:100]}...")
    
    # Stage 2: Context Selection
    context = "".join([f"\n[Source: {r['chunk']['source']}] {r['chunk']['text']}\n" for r in results])
    
    # Stage 3: Prompt Construction
    final_prompt = f"""Answer based ONLY on context. If not found, say you don't know.\nDocuments: {context}\nQuery: {query}\nAnswer:"""
    if verbose:
        print("--- FINAL PROMPT SENT TO LLM ---")
        print(final_prompt[:500] + "...")
        print("--------------------------------")
    
    # Stage 4: LLM Response
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": final_prompt}],
        temperature=0
    )
    return response.choices[0].message.content

print("✅ Pipeline implementation complete.")

### Part D Test: Normal Query
**Query:** "What was the total expenditure in 2024?"

In [ ]:
normal_query = "What was the total expenditure in 2024?"
answer = rag_pipeline(normal_query)
print(f"\n--- FINAL RESPONSE ---\n{answer}")

---
## 🔴 PART E: CRITICAL EVALUATION & ADVERSARIAL TESTING
**Objective:** Test the system against ambiguous and misleading queries and compare it with a Pure LLM.

In [ ]:
def pure_llm_response(query):
    res = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": query}],
        temperature=0
    )
    return res.choices[0].message.content

adversarial_queries = [
    {"type": "Ambiguous", "query": "Who won the election?"},
    {"type": "Misleading", "query": "How many votes did the NPP get in the 2025 budget?"}
]

for test in adversarial_queries:
    print(f"\n{'='*50}")
    print(f"TEST TYPE: {test['type']}")
    print(f"QUERY: {test['query']}")
    print(f"{'='*50}")
    
    print("\n--- PURE LLM RESPONSE ---")
    print(pure_llm_response(test['query']))
    
    print("\n--- RAG SYSTEM RESPONSE ---")
    print(rag_pipeline(test['query'], verbose=False))